# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayyanA24/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
## Method Choice

I chose a Decision Tree Classifier because it is simple, interpretable, and works well as a baseline machine learning model. It can capture interactions between features better than the hand-written scoring rule while remaining easy to explain.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
## Split Design

I used a random train-test split with 80% of the data for training and 20% for testing. This provides an honest evaluation because the model is tested on data that it did not see during training.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "search_volume",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update"
]

X = df[features].fillna(0)

y = (df["trend_direction"] == "down").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


(24000, 7)
(6000, 7)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
## Model Comparison

The Decision Tree model is compared with the baseline rule using the same train-test split. Accuracy is used only as a simple comparison metric. The model should be interpreted as decision-support rather than an automatic decision maker.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import accuracy_score
import pandas as pd

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

model_acc = accuracy_score(y_test, pred)

baseline_pred = (
    X_test["impressions_90d"]
    > X_test["impressions_90d"].median()
).astype(int)

baseline_acc = accuracy_score(y_test, baseline_pred)

comparison = pd.DataFrame({
    "Model": ["Baseline Rule", "Decision Tree"],
    "Accuracy": [baseline_acc, model_acc]
})

print(comparison)


           Model  Accuracy
0  Baseline Rule  0.553333
1  Decision Tree  0.631667


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
## Errors and Interpretation

The model sometimes misclassifies pages whose feature values are close to the decision boundary. This is expected because content performance depends on many factors that are not included in the starter dataset.

The Decision Tree mainly relies on historical search metrics and content characteristics. The predictions should be treated as recommendations for review rather than final decisions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

print("Confusion Matrix")

print(confusion_matrix(y_test, pred))

print("\nFeature Importance")

importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)


Confusion Matrix
[[1281 1467]
 [ 743 2509]]

Feature Importance
impressions_90d           0.575653
content_age_days          0.257441
avg_position              0.098087
ctr                       0.068818
search_volume             0.000000
word_count                0.000000
days_since_last_update    0.000000
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.